# 🛣️ ROADGUARD AI — HUẤN LUYỆN MODEL YOLOv11 SIÊU QUY MÔ 44,000 ẢNH
### Nhận Diện Ổ Gà (Pothole) + Vết Nứt (Crack) + Mẫu Âm Tính Chống Báo Động Giả
---
> **Dataset:** `unified_road_dataset_full.zip` (43,940 ảnh + 43,940 file nhãn `.txt`)
> **2 Classes:** `pothole` (0), `crack` (1) + 20,000 ảnh nền mặt đường phẳng (Background/Negative)
> **Phần cứng:** Google Colab GPU Tesla T4 (Miễn phí)
> **Đầu ra:** File trọng số xuất xưởng `roadguard_best_44k.pt`

## 📌 Bước 1: Kiểm Tra GPU T4 & Cài Đặt Thư Viện

In [ ]:
# 1. Kiểm tra trạng thái GPU
!nvidia-smi

import torch
print("=" * 65)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM Dung lượng:{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào menu Runtime -> Change runtime type -> Chọn T4 GPU")
print("=" * 65)

# 2. Cài đặt Ultralytics mới nhất
!pip install -q ultralytics

## 📥 Bước 2: Tự Động Nhận Diện & Giải Nén File `unified_road_dataset_full.zip`
---
> **Cách làm:** Bạn chỉ cần bấm vào biểu tượng thư mục 📁 (Files) ở mép trái của Google Colab, sau đó KÉO THẢ file `unified_road_dataset_full.zip` (453MB) vào đó.

In [ ]:
import os, glob, zipfile, sys
from pathlib import Path

dataset_dir = "/content/dataset"
os.makedirs(dataset_dir, exist_ok=True)

# 1. Quét tìm file zip dataset trong Colab
zip_files = glob.glob("/content/*.zip") + glob.glob("*.zip")
print(f"🔍 Tìm thấy các file zip trong môi trường: {zip_files}")

if not zip_files:
    raise FileNotFoundError("❌ CHƯA THẤY FILE ZIP! Bạn hãy kéo thả file 'unified_road_dataset_full.zip' vào cột Files bên trái Colab rồi chạy lại ô này.")

# Ưu tiên chọn file unified_road_dataset_full.zip
chosen_zip = next((z for z in zip_files if "full" in z.lower()), zip_files[0])
print(f"📦 Đang giải nén tập dữ liệu: {chosen_zip}...")

with zipfile.ZipFile(chosen_zip, 'r') as z:
    z.extractall(dataset_dir)
print("🎉 GIẢI NÉN THÀNH CÔNG TOÀN BỘ DATASET!")

# 2. Định vị file data.yaml
yaml_candidates = glob.glob(f"{dataset_dir}/**/data.yaml", recursive=True)
if not yaml_candidates:
    # Tự động tạo data.yaml nếu giải nén ra thẳng train/valid/test
    yaml_path = f"{dataset_dir}/data.yaml"
    with open(yaml_path, 'w') as f:
        f.write(f"path: {dataset_dir}\ntrain: train/images\nval: valid/images\ntest: test/images\nnc: 2\nnames: ['pothole', 'crack']\n")
else:
    yaml_path = os.path.abspath(yaml_candidates[0])

# Cập nhật đường dẫn tuyệt đối vào data.yaml để tránh lỗi đường dẫn tương đối của Colab
root_extracted = str(Path(yaml_path).parent).replace('\\', '/')
fixed_yaml_content = f"""path: {root_extracted}
train: train/images
val: valid/images
test: test/images

nc: 2
names: ['pothole', 'crack']
"""
with open(yaml_path, 'w') as f:
    f.write(fixed_yaml_content)

print("=" * 65)
print(f"✅ Đã cấu hình data.yaml chuẩn xác tại: {yaml_path}")
print(fixed_yaml_content)

# 3. Thống kê số lượng ảnh thực tế
train_cnt = len(glob.glob(f"{root_extracted}/train/images/*.*"))
val_cnt = len(glob.glob(f"{root_extracted}/valid/images/*.*"))
test_cnt = len(glob.glob(f"{root_extracted}/test/images/*.*"))
print(f"📊 Số lượng ảnh sẵn sàng: Train={train_cnt:,} | Valid={val_cnt:,} | Test={test_cnt:,} | TỔNG={train_cnt + val_cnt + test_cnt:,} ảnh!")
print("=" * 65)

## 🧠 Bước 3: Huấn Luyện Mô Hình YOLOv11 (2 Classes: Pothole & Crack)

In [ ]:
from ultralytics import YOLO

# Khởi tạo kiến trúc YOLOv11 Nano (chuyên dụng cho suy luận thời gian thực trên Drone/Edge AI)
model = YOLO('yolo11n.pt')

print("🚀 BẮT ĐẦU HUẤN LUYỆN YOLOv11 TRÊN 44,000 ẢNH VỚI GPU T4...")
results = model.train(
    data=yaml_path,
    epochs=50,             # 50 epochs để hội tụ hoàn hảo cả 2 lớp
    imgsz=640,             # Kích thước chuẩn lưới 640x640 của RoadGuard Tiling Engine
    batch=16,              # Kích thước batch tối ưu cho VRAM 15GB T4
    device=0,              # GPU CUDA
    workers=4,
    save=True,
    save_period=10,        # Lưu checkpoint mỗi 10 epochs
    plots=True,
    name='roadguard_44k_yolo11'
)

print("🎉 HUẤN LUYỆN TOÀN BỘ 50 EPOCHS THÀNH CÔNG!")

## 📊 Bước 4: Trực Quan Hóa Chỉ Số Đánh Giá (Loss, mAP50, Confusion Matrix)

In [ ]:
from IPython.display import Image, display

run_dir = "runs/detect/roadguard_44k_yolo11"

# 1. Biểu đồ hội tụ hàm mất mát và độ chính xác mAP50
results_img = f"{run_dir}/results.png"
if os.path.exists(results_img):
    print("📈 1. Biểu đồ đường cong học tập (Learning Curves):")
    display(Image(results_img))

# 2. Ma trận nhầm lẫn (Confusion Matrix)
cm_img = f"{run_dir}/confusion_matrix.png"
if os.path.exists(cm_img):
    print("🎯 2. Ma trận nhầm lẫn (Confusion Matrix - Phân biệt Pothole vs Crack vs Background):")
    display(Image(cm_img))

# 3. Ảnh kết quả dự đoán trên tập kiểm thử
val_preds = glob.glob(f"{run_dir}/val_batch*_pred.jpg")
if val_preds:
    print("🔍 3. Mẫu ảnh dự đoán trực quan trên tập kiểm thử:")
    display(Image(val_preds[0]))

## 💾 Bước 5: Xuất Xưởng & Tải File Trọng Số `roadguard_best_44k.pt` Về Máy

In [ ]:
import shutil
from google.colab import files

src_weight = f"{run_dir}/weights/best.pt"
out_weight_name = "roadguard_best_44k.pt"

if os.path.exists(src_weight):
    shutil.copy(src_weight, out_weight_name)
    print(f"✅ Đã xuất xưởng mô hình tốt nhất: {out_weight_name}")
    print("⬇️ Đang kích hoạt tải file về máy tính của bạn...")
    files.download(out_weight_name)
    print("=" * 65)
    print(f"👉 Sau khi tải xong, hãy chép file vào:\n   F:\\train AI đồ án\\weights\\trained\\{out_weight_name}")
    print("=" * 65)
else:
    print(f"❌ Không tìm thấy file {src_weight}")